###  Total number of encounters by year, quarter, and encounter class


In [0]:

SELECT 
  c.year,
  c.quarter,
  f.encounter_class,

  COUNT(*) as total,

  ROUND(
    COUNT(*) * 100.0 / 
    SUM(COUNT(*)) OVER (PARTITION BY c.year, c.quarter),
    2
  ) as percentage

FROM medical_catalog.gold.fact f
JOIN medical_catalog.gold.dim_calendar c 
  ON DATE(f.start_time) = c.date

GROUP BY c.year, c.quarter, f.encounter_class
ORDER BY c.year, c.quarter;


### How long Patient stay (Under 24 vs Over 24)

In [0]:
SELECT 
  c.year,
  c.month,

  CASE 
    WHEN f.duration_hours > 24 THEN 'Over 24 hrs'
    ELSE 'Under 24 hrs'
  END as type,

  COUNT(*) as total,

  ROUND(
    COUNT(*) * 100.0 / 
    SUM(COUNT(*)) OVER (PARTITION BY c.year, c.month),
    2
  ) as percentage

FROM medical_catalog.gold.fact f
JOIN medical_catalog.gold.dim_calendar c 
  ON DATE(f.start_time) = c.date

GROUP BY c.year, c.month, type
ORDER BY c.year, c.month;


### Zero Payer Coverage

In [0]:
SELECT 
  c.year,
  c.month,
  p.payer_name,

  COUNT(*) as total_encounters,

  -- Zero coverage
  SUM(
    CASE 
      WHEN f.payer_id IS NULL OR f.total_claim_cost = 0 
      THEN 1 ELSE 0 
    END
  ) as zero_coverage,

  -- % zero coverage
  ROUND(
    SUM(CASE 
          WHEN f.payer_id IS NULL OR f.total_claim_cost = 0 
          THEN 1 ELSE 0 
        END) * 100.0 / COUNT(*),
    2
  ) as zero_coverage_pct,

  -- % covered
  ROUND(
    SUM(CASE 
          WHEN f.total_claim_cost > 0 
          THEN 1 ELSE 0 
        END) * 100.0 / COUNT(*),
    2
  ) as coverage_pct

FROM medical_catalog.gold.fact f
JOIN medical_catalog.gold.dim_calendar c 
  ON DATE(f.start_time) = c.date
LEFT JOIN medical_catalog.gold.dim_payer p 
  ON f.payer_id = p.payer_id

GROUP BY c.year, c.month, p.payer_name
ORDER BY c.year, c.month;


### Top 10 Procedures

In [0]:
WITH aggregated AS (
  SELECT
    c.year,
    c.half_year,
    p.procedure_code,
    ROUND(AVG(f.base_cost),2) AS avg_cost,
    COUNT(*) AS performed_count
  FROM medical_catalog.gold.fact f
  JOIN medical_catalog.gold.dim_calendar c
    ON DATE(f.start_time) = c.date
  JOIN medical_catalog.gold.dim_procedure p
    ON f.encounter_id = p.encounter_id
  GROUP BY c.year, c.half_year, p.procedure_code
)
SELECT *
FROM (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY year, half_year
      ORDER BY avg_cost DESC
    ) AS rn
  FROM aggregated
)
WHERE rn <= 10

### Avg_Total_Claim

In [0]:
SELECT 
  p.payer_name,
  ROUND(AVG(f.total_claim_cost),2) as avg_cost

FROM medical_catalog.gold.fact f
JOIN medical_catalog.gold.dim_payer p 
  ON f.payer_id = p.payer_id

GROUP BY p.payer_name
ORDER BY avg_cost DESC;


### Readmission_Rate

In [0]:
WITH seq AS (
  SELECT 
    pateint_id,
    encounter_id,
    start_time,
    end_time,

    LAG(end_time) OVER (
      PARTITION BY pateint_id 
      ORDER BY start_time
    ) as prev_end

  FROM medical_catalog.gold.fact
)

, flags AS (
  SELECT *,

    CASE 
      WHEN prev_end IS NOT NULL 
           AND start_time >= prev_end
      THEN 1 ELSE 0 
    END as eligible,

    CASE 
        WHEN prev_end IS NOT NULL AND start_time >= prev_end 
           AND DATEDIFF(start_time, prev_end) <= 30
      THEN 1 ELSE 0 
    END as readmission

  FROM seq
)
SELECT 
  c.year,
  c.month,

  SUM(eligible) as eligible,
  SUM(readmission) as readmissions,

  ROUND(
    SUM(readmission) * 100.0 / SUM(eligible),
    2
  ) as rate

FROM flags f
JOIN medical_catalog.gold.dim_calendar c 
  ON DATE(f.start_time) = c.date

GROUP BY c.year, c.month
ORDER BY c.year, c.month;


